# Fine-tuning DistilBERT for Extractive Question Answering Using Stanford Question Answering Dataset with Sliding Windows Preprocessing

**Pablo Apausa**  
Deep Learning For Natural Language Processing  

Universidad Politécnica de Madrid  
_Noviembre, 2025_  

----

## 1. Introducción

### 1.1. Extractive QA

Este notebook demuestra el fine-tuning de DistilBERT para el *extractive question-answering* usando el Stanford Question Answering Dataset (SQuAD) y preprocesamiento con ventanas deslizantes. A diferencia de los enfoques generativos que crean texto, esta tarea identifica y extrae el fragmento exacto del texto a partir del contexto en el que se enmarca una pregunta.

### 1.2. DistilBERT

Se ha seleccionado DistilBERT en lugar de modelos como BERT por razones de eficiencia computacional: al desarrollar desde cero es importante valorar la capacidad para iterar rápido y hacer muchas pruebas. 

DistilBERT es una versión destilada que retiene el 97% del rendimiento de BERT mientras reduce el número de parámetros en un 40% y aumenta la eficiencia en un 60% **(Sanh et al., 2020)**. Esta optimización permite entrenar el modelo de manera más rápida sin sacrificar calidad en las predicciones.

### 1.3. Fine-tuning

El *fine-tuning* consiste en adaptar un modelo preentrenado a una tarea específica para mejorar su rendimiento. En este caso, mientras que el preentrenamiento le proporciona al modelo una comprensión general del lenguaje, el *fine-tuning* lo especializa para identificar las posiciones de inicio y fin de las respuestas dentro de un contexto dado.

### 1.4. Índice

El notebook se estructura en las siguientes fases:
- **3. Preparación de los datos**: SQuAD, preprocesamiento con ventanas deslizantes, mantenimiento de alineación de carácter a token y generación de etiquetas a nivel de token.
- **4. Entrenamiento**: Ajuste fino de DistilBERT para predecir fragmentos de respuesta sobre 87,735 características en aproximadamente 40.5 minutos.
- **5. Evaluación**: Posprocesamiento de predicciones, agregación de características, cálculo de métricas de SQuAD (73.59% EM, 82.57% F1).
- **6. Análisis de Rendimiento**: Demostración de mejora sobre el modelo base en preguntas específicas sobre las diapositivas de clase.

----

## 2. Configuración Inicial

### 2.1 Comprobación de la GPU

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 2.2 Instalación de las librerías

- transformers: proporciona modelos transformer preentrenados como DistilBERT y tokenizadores.
- datasets: permite la carga y procesamiento eficiente de SQuAD.
- evaluate: contiene las métricas de evaluación que miden la calidad de las predicciones.
- protobuf==3.20.3: asegura la compatibilidad con otras bibliotecas.

In [2]:
pip install -q transformers datasets evaluate protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 42.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requi

## 3. Preparación de los datos

### 3.1 SQuAD Dataset

El Stanford Question Answering Dataset (SQuAD) es un punto de referencia para la *extractive question-answering*. Siendo uno de los más usados para casos de uso pequeños como este. Este consta de más de 100.000 preguntas planteadas por colaboradores sobre un conjunto de artículos de Wikipedia, donde la respuesta a cada pregunta es un segmento del pasaje correspondiente **(Rajpurkar et al., 2016)**^. 

La estructura del conjunto de datos incluye tres campos clave: `context`, `question` y `answers`; estás últimas son un diccionario con `text` y `answer_start` (el índice de carácter donde comienza la respuesta). Esta posición de carácter es crucial para convertir a posiciones de token durante el entrenamiento.

Los ejemplos de entrenamiento contienen exactamente una respuesta por pregunta, mientras que los ejemplos de validación pueden tener múltiples respuestas aceptables de diferentes anotadores. Durante la evaluación, las predicciones se comparan contra todas las respuestas válidas y se toma la mejor puntuación.

In [3]:
from datasets import load_dataset

# Load SQuAD dataset: contains context, question, and answer (with character position)
# Returns DatasetDict with 'train' and 'validation' splits
squad = load_dataset("squad")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

### 3.2 Declaración del tokenizador

Los transformer como DistilBERT requieren que el texto se convierta en token numéricos. El tokenizador gestiona esta conversión preservando las relaciones semánticas entre palabras y subpalabras.

Para la respuesta de preguntas, el tokenizador combina la pregunta y el contexto en una sola secuencia usando tokens especiales: [CLS] al inicio, [SEP] separando la pregunta del contexto, y [SEP] al final. Esta estructura permite a DistilBERT distinguir la pregunta del contexto.

El tokenizador distilbert-base-cased es sensible a mayúsculas y minúsculas. También proporciona mapeos de desplazamiento que vinculan cada token con su posición en el texto original, lo que permite convertir las posiciones de las respuestas de caracteres a tokens para el entrenamiento.

In [ ]:
from transformers import AutoTokenizer

# load distilBERT
model_name = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

### 3.3 Preprocesamiento del Conjunto de Datos de Entrenamiento

El preprocesamiento de los datos de entrenamiento es la parte más compleja del proceso, ya que se deben gestionar contextos largos y generar etiquetas para las posiciones de las respuestas. El reto es que muchos contextos en SQuAD superan la longitud máxima de entrada de 320. 

En lugar de truncar y perder información, se emplea un enfoque de ventana deslizante: dividiendo los contextos largos en fragmentos superpuestos con un paso de 128 tokens. Esto garantiza si una respuesta se acerca a un punto de truncamiento, aparecerá completamente dentro de al menos un fragmento.

Esta función de preprocesamiento debe mapear las posiciones de las respuestas a los índices dentro de cada fragmento. Esto se logra mediante asignaciones de desplazamiento: una lista de tuplas `(start_char, end_char)`que muestra qué intervalo de caracteres en el texto original representa cada token. 

El proceso de etiquetado tokeniza los pares pregunta-contexto, asigna una etiqueta nula cuando las respuestas caen fuera de los fragmentos, y de lo contrario encuentra el primer y último token del fragmento para crear las etiquetas `start_positions` y `end_positions`. Además, se completan todas las secuencias hasta la longitud máxima para un procesamiento por lotes eficiente. 

In [ ]:
max_length = 320 # maximum sequence length
stride = 128 # sliding window stride for handling long contexts

def preprocess_training(examples):
    questions = [q.strip() for q in examples["question"]] # strip extra whitespace from questions
    
    # tokenize question-context pairs
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second", # keep questions intact, split contexts
        stride=stride,
        return_overflowing_tokens=True, # create multiple features from long contexts
        return_offsets_mapping=True, # get character-to-token alignment
        padding="max_length",
    )

    # extract offset mappings and sample mappings for label generation
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    # generate start and end position labels for each feature
    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # if answer is not fully inside the context chunk, label as (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # find token indices for answer start and end using offset mappings
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

Este preprocesamiento se aplica usando `Dataset.map()` con `batched=True` para eficiencia y `remove_columns=squad["train"].column_names` para reemplazar la estructura original con entradas tokenizadas y etiquetas de posición.

In [ ]:
# Apply preprocessing to training set
train_dataset = squad["train"].map(
    preprocess_training,
    batched=True, # process multiple examples at once for efficiency
    remove_columns=squad["train"].column_names, # replace original structure with tokenized inputs and labels
)

len(squad["train"]), len(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

(87599, 87735)

El preprocesamiento aumenta el dataset de 87599 a 88729 ejemplos. Es decir, se han creado 1130 muestras adicionales. 

### 3.4 Preprocesamiento del Conjunto de Datos de Validación

Preprocesar el conjunto de validación es más sencillo, ya que no se generan etiquetas de respuesta, si no que el modelo las predice durante la evaluación. Sin embargo, se debe preservar la información para mapear las predicciones de vuelta a los ejemplos originales y extraer el texto de la respuesta.

El principal desafío durante la evaluación radica en que el modelo genera predicciones a nivel de token, pero se necesita reportar la extensión del contexto original. Además, cuando los contextos largos se dividen en múltiples características, se deben agregar para determinar la mejor respuesta para el ejemplo original.

Así que para solucionarlo, se mantienen las asignaciones de desplazamiento, pero con una modificación: se anulan los *offset* de los *tokens* de preguntas. Y de esta manera, se consideran únicamente los *tokens* de contexto al extraer los intervalos de respuestas.

Los IDs de ejemplo se preservan porque las ventanas deslizantes crean múltiples características por cada uno, y `overflow_to_sample_mapping` rastrea de qué ejemplo original provino cada característica.

In [ ]:
def preprocess_validation(examples):
    questions = [q.strip() for q in examples["question"]]  # strip extra whitespace from questions

    # same as training, but no labels needed
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        ids.append(examples["id"][sample_idx]) # store example ID to group features from same example during evaluation

        # set question token offsets to None, keeping only context token offsets
        # this helps identify valid answer positions during post-processing
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = ids
    return inputs

Este preprocesamiento se aplica al conjunto de validación, reemplazando las columnas originales con características procesadas que contienen entradas tokenizadas, mapeos de desplazamiento e IDs de ejemplo.

In [ ]:
# apply preprocessing to validation set
validation_dataset = squad["validation"].map(
    preprocess_validation,
    batched=True,
    remove_columns=squad["validation"].column_names,
)

len(squad["validation"]), len(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(10570, 10629)

El preprocesamiento de validación ha añadido 59 características, lo que sugiere que los contextos del conjunto de validación suelen ser más cortos que los del conjunto de entrenamiento y requieren menos división. 

----

## 4. Implementación

### 4.1 Inicialización del Modelo

El modelo base de DistilBERT proporciona representaciones contextualizadas para cada token en la entrada, mientras que el módulo de preguntas y respuestas añade dos capas lineales: una que predice la probabilidad de que cada token sea el inicio y otra que predice la probabilidad de que cada token sea el final de la respuesta.

 La advertencia sobre los pesos recién inicializados es de esperar, ya que estas capas de la tarea se inicializan aleatoriamente mientras que las capas preentrenadas conservan la comprensión del lenguaje. El ajuste fino entrena ambos componentes para la extracción de respuestas.

In [ ]:
from transformers import AutoModelForQuestionAnswering

# initialize distilBERT with qa head
model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

2025-11-25 18:42:01.421724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764096121.615372      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764096121.668881      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 4.3 Definición de los hiperparámetros

`TrainingArguments` encapsula los hiperparámetros que controlan el proceso de entrenamiento

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-squad", # output directory for checkpoints
    eval_strategy="epoch", # evaluate at end of each epoch
    save_strategy="epoch", # save checkpoint at end of each epoch
    learning_rate=2e-5, # standard learning rate for BERT fine-tuning
    num_train_epochs=1, # one pass through dataset
    weight_decay=0.01, # regularization to prevent overfitting
    fp16=True, # mixed precision training for speed
    report_to="none", # disable logging
)

### 4.4 Creación del Entrenador

Trainer maneja el bucle de entrenamiento, la configuración del optimizador y la evaluación. Cada paso procesa un *batch*: *forward pass*, cálculo de pérdida, retropropagación y actualizaciones de pesos.

In [11]:
from transformers import Trainer

# trainer orchestrates the training loop, gradient computation, and optimization
trainer = Trainer(
    model=model, # distilBERT with qa head
    args=args, # training configuration
    train_dataset=train_dataset, # preprocessed training features with labels
    eval_dataset=validation_dataset, # preprocessed validation features
    processing_class=tokenizer, # tokenizer for batch collation
)

### 4.5 Ejecución del *fine-tuning*

El entrenamiento se inicia con `trainer.train()`, procesando 87735 características a lo largo de 10967 pasos por epoch. Su entrenamiento tarda  alrededor de 40 minutos. 

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.208900,No log


TrainOutput(global_step=10967, training_loss=1.5280161273504462, metrics={'train_runtime': 2428.2934, 'train_samples_per_second': 36.13, 'train_steps_per_second': 4.516, 'total_flos': 1.146284973130752e+16, 'train_loss': 1.5280161273504462, 'epoch': 1.0})

----

## 5. Resultados y discusión

### 5.1. Evaluación

La evaluación convierte las predicciones del modelo a fragmentos de texto y maneja múltiples características por ejemplo. La estrategia considera las 20 primeras posiciones del principio y las 20 primeras posiciones del final, evalúa combinaciones válidas y puntúa cada una sumando los logits. 

Las respuestas que se filtran son aquellas fuera del contexto (mediante desplazamientos `None`), con una longitud negativa, o con más de 30 tokens.  Para los contextos divididos, se selecciona la respuesta con la puntuación más alta. Si no se encuentran perspectivas válidas para un ejemplo, se devuelve una cadena vacía como predicción.

La evaluación usa dos métricas complementarias. Primero está la coincidencia exacta (EM), que mide si el modelo identifica con precisión el fragmento de contexto: dando una puntuación de 1 a respuestas exactas y de 0 en caso contrario. 

Y la puntuación F1, que calcula la media armónica de precisión entre la predicción y el *ground-truth*. Es más flexible que el EM, otorgando una puntuación parcial cuando hay coincidencia entre la respuesta predicha y la correcta.


In [ ]:
from tqdm.auto import tqdm
import collections
import numpy as np
import evaluate

n_best = 20  # consider top 20 start/end positions
max_answer_length = 30  # maximum answer span length in tokens
metric = evaluate.load("squad") # load SQuAD metric for evaluation (exact match and f1)

def check_score(start_logits, end_logits, features, examples):
    # map each example to its features
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    
    # process each example to find best answer across all its features
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # collect candidate answers from all features for this example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            # get top 20 start and end positions by logit score
            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            
            # Eealuate all valid (start, end) combinations
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # skip if answer is not fully in context (offset is None for question tokens)
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # skip invalid answers: negative length or too long
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    # extract answer text from context using offsets
                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # select answer with highest logit score (or empty string if no valid answers)
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    # format ground truth answers and compute SQuAD metrics (em and f1)
    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

Ahora después del entrenamiento, `predict()` ejecuta inferencias sobre todo el *dataset* de validación, y devuelve las predicciones del modelo. Estos logits iniciales y finales para cada token de cada característica se pasan a la función de antes. 

In [ ]:
# run inference on validation set and get start/end logits
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions

# post-process predictions to extract answer spans and compute metrics
check_score(start_logits, end_logits, validation_dataset, squad["validation"])

  0%|          | 0/10570 [00:00<?, ?it/s]

{'exact_match': 73.58561967833491, 'f1': 82.57144442016161}

#### Muy bien! 😃

Se alcanza 73.59% de EM y 82.57% de F1 en la validación del *fine-tuning*. 

### 5.2. Comparación

Para demostrar la mejora, el DistilBERT base se compara con la versión *fine-tuned*, siguiendo el ejemplo del notebook "A Simplified Guide to Fine-Tuning an Open LLM with QLoRA" **(Serrano, 2025)**. Le proporciono casos de prueba en base a las diapositivas de clase para ver si acierta.

In [ ]:
from transformers import pipeline

# create qa pipeline for base model (no fine-tuning)
base_qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-cased"
)

# create qa pipeline for fine-tuned model
tuned_qa_pipeline = pipeline(
    task="question-answering",
    model=trainer.model,
    tokenizer=tokenizer
)

# test cases
test_cases = [
    {
        "context": """
        Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, is designed to LLMs with additional information from an external knowledge source.
        The main purpose of RAG is to generate more accurate and contextual answers while reducing hallucinations.
        In RAG, factual knowledge is separated from the LLMs reasoning capability and stored in an external knowledge source.
        This technique is analogous to taking an open-book exam: "RAG is to LLMs what an open-book exam is to humans".
        """,
        "question": "What is the primary function of Retrieval-Augmented Generation (RAG) in LLM applications?"
    },
    {
        "context": """
        GPT-3 is an autoregressive language model released by OpenAI in 2020.
        This model is characterized by its scale, which includes 96 Transformer decoder layers.
        The model possesses 175 billion parameters and was trained using 300 billion tokens from several sources.
        Furthermore, GPT-3 has a 2048-token-long context window, which represents the amount of text the model can consider when generating a response.
        """,
        "question": "How many parameters does the GPT-3 model have?"
    }, 
    {
        "context": """
        The transformer architecture was introduced by Vaswani et al. in the paper "Attention Is All You Need" (2017).
        The architecture eliminates Recurrent Neural Networks (RNNs) and uses only attention mechanisms.
        The architecture especially improves the training speed because it deals with sequential data in parallel. 
        Language Models like Gemini and ChatGPT are based on transformers.
        """,
        "question": "Which research paper introduced the Transformer architecture?"
    },
    {
        "context": """
        Long short-term memory (LSTM) models were developed to address long-term dependencies, solving issues found in traditional RNNs that often exhibit short-term memory.
        LSTMs introduce the cell state, which is key to their functionality.
        The cell state acts as the long-term memory of the LSTM, designed to carry information across many time steps while undergoing minimal modifications.
        In contrast, the hidden state represents the short-term, immediate output, used for predictions and communication with other layers.
        """,
        "question": "What role does the cell state play within the Long short-term memory (LSTM) architecture?"
    },
    {
        "context": """
        Word embeddings are designed not only to reflect similarity but also to preserve semantic and syntactic relationships.
        These relationships are observed in the difference vectors between words.
        This property allows for analogies or queries to be expressed as equations
        Furthermore, these sophisticated relationships can be leveraged in other applications, such as assisting with machine translation.
        """,
        "question": "How do word embeddings encode semantic and syntactic relationships between words?"
    },
        {
        "context": """
        Skip-gram predicts a target context from a word.
        For example, from ('mat'), it predicts 'the cat sits on the'.
        Predicting a target context from a word is a fake task!.
        The goal is actually just to learn the weight matrix of the hidden layer (word embeddings).
        """,
        "question": "If predicting a target context is a fake task, what is the actual goal of Skip-gram?"
    }, 
]

# compare base and fine-tuned model on each test case
for i, test_case in enumerate(test_cases, 1):
    context = test_case["context"].strip()
    question = test_case["question"]

    base_result = base_qa_pipeline(question=question, context=context)
    base_answer = base_result["answer"]
    base_score = base_result["score"]

    tuned_result = tuned_qa_pipeline(question=question, context=context)
    tuned_answer = tuned_result["answer"]
    tuned_score = tuned_result["score"]

    print(f"Test case #{i}")
    print(f"Context: {context}")
    print(f"Question: {question}")
    print()
    
    print("Base model:")
    print(f"- Answer: {base_answer}")
    print(f"- Confidence: {base_score:.4f}")
    print()
    
    print("Fine-tuned model:")
    print(f"- Answer: {tuned_answer}")
    print(f"- Confidence: {tuned_score:.4f}")
    print()
    print()


Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Device set to use cuda:0


Test case #1

Context: Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, is designed to provide Large Language Models (LLMs) with additional information from an external knowledge source.
        The main purpose of RAG is to generate more accurate and contextual answers while reducing hallucinations.
        In RAG, factual knowledge is separated from the LLMs reasoning capability and stored in an external knowledge source.
        This technique is analogous to taking an open-book exam: "RAG is to LLMs what an open-book exam is to humans".

Question: What is the primary function of Retrieval-Augmented Generation (RAG) in LLM applications?

Base model:

- Answer: .
        This technique is analogous to taking an open-

- Confidence: 0.0002

Fine-tuned model:

- Answer: to generate more accurate and contextual answers while reducing hallucinations

- Confidence: 0.2071

Test case #2

Context: GPT-3 is an autoregressive language model released by OpenAI in 2020.

----

## 6. Conclusion y pasos futuros

Por una parte, los resultados obtenidos demuestran un buen rendimiento considerando que solo se ha entrenado por dos epochs.

Por otra parte, la comparación con el modelo base en la sección anterior demuestra la eficacia del *fine-tuning*. Este primer modelo tiene un comportamiento casi aleatorio, con niveles de confianza muy bajos y respuestas incoherentes que extraen fragmentos sin sentido. Mientras que el modelo con *fine-tuning* demuestra:
1. Una confianza mucho mayor: reflejando predicciones más decisivas.
2. Coherencia semántica: las respuestas son apropiadas y completas, como "175 billion" para el número de parámetros de GPT-3. 
3. Y capacidad de generalización: el modelo responde a preguntas sobre conceptos de clase no presentes en el *dataset*, lo que dejar ver que ha aprendido patrones generales de comprensión lectora en lugar de memorizar ejemplos.

Esta recorrido de un comportamiento aleatorio hasta predicciones útiles y confiables en el mismo modelo muestra el valor del *fine-tuning*. Aún cuando se realiza sobre un modelo pequeño como DistilBERT. Los resultados validan que unas pocas épocas de entrenamiento son suficiente para adaptar el conocimiento lingüístico preentrenado a la tarea de respuesta extractiva. 

Como pasos futuros, sería interesante probar a entrenar modelos más grandes como `bert-base-cased` durante más epochs, comparando los resultados de EM y F1 con este modelo. 

----

## Bibliografía

- Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*. arXiv preprint arXiv:1910.01108.

- Rajpurkar, P., Zhang, J., Lopyrev, K., & Liang, P. (2016). *Squad: 100,000+ questions for machine comprehension of text*. arXiv preprint arXiv:1606.05250.

- Serrano, E. (2025). *A Simplified Guide to Fine-Tuning an Open LLM with QLoRA*. Retrieved from https://moodle.upm.es/titulaciones/oficiales/pluginfile.php/13931263/mod_folder/content/0/GenAILMs_QLORA.ipynb